<a href="https://colab.research.google.com/github/Mustafa0291polat/AI-Arac-Kiralama-Platformu/blob/feature%2FKAN-12-ai-api-baglanti-testleri/Yaz%C4%B1l%C4%B1m_M%C3%BCh.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# @title OpenAI
import json
from openai import OpenAI
from google.colab import userdata

client = OpenAI(api_key=userdata.get('OPENAI'))

def filtreleme_uygun_arac(kullanici_metni):
    print("OpenAI API'sine istek gönderiliyor..")
    try:
        # API İsteği
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {
                    "role": "system",
                    "content": """Sen "Yapay Zeka Destekli Bireyler Arası Araç Kiralama Platformu" için çalışan bir asistansın.
Görevin, kullanıcının girdiği serbest metni analiz ederek, sadece JSON formatında filtreler çıkarmaktır. Başka hiçbir açıklama yazma.
Format: {"arac_tipi": "SUV", "min_koltuk_sayisi": 5, "ozellikler": ["geniş bagaj", "arazi"], "butce_tercihi": "ekonomik"}"""
                },
                {
                    "role": "user",
                    "content": kullanici_metni
                }
            ],
            max_tokens=150,
            # düşük değerler modelin daha tutarlı ve formata uygun cevap vermesini sağlıyor.
            temperature=0.2
        )

        ham_yanit = response.choices[0].message.content

        # Görev 3: Veriyi Parse Etme / Filtrelemeye göre veriler arttırılacak
        islenmis_veri = json.loads(ham_yanit)

        print("✅ Başarılı! İşlenmiş Veri / Veritabanı Araması İçin Hazır:")
        print(f"-> Araç Tipi: {islenmis_veri.get('arac_tipi')}")
        print(f"-> Minimum Koltuk: {islenmis_veri.get('min_koltuk_sayisi')}")
        print(f"-> Özellikler: {', '.join(islenmis_veri.get('ozellikler', []))}")
        print(f"-> Bütçe: {islenmis_veri.get('butce_tercihi')}")

        return islenmis_veri

    # Görev 3: Hata Yönetimi
    # Model JSON formatını bozarsa sistemin çökmesini engellemek için
    except json.JSONDecodeError:
        print("❌ Hata: API'den dönen yanıt geçerli bir JSON formatında değil.")
        print("Gelen bozuk yanıt:", ham_yanit)
        return None
    # API'nin çökmesi, internetin gitmesi vs. durumlar için
    except Exception as e:
        print(f"❌ Beklenmeyen bir API hatası oluştu: {e}")
        return None

# Test Senaryosunu
test_metni = "Bu hafta sonu 5 kişilik ailemle kampa gideceğim, dağ yollarına uygun, geniş bagajlı ve bütçe dostu bir araç arıyorum"
sonuc = filtreleme_uygun_arac(test_metni)

OpenAI API'sine istek gönderiliyor..
✅ Başarılı! İşlenmiş Veri / Veritabanı Araması İçin Hazır:
-> Araç Tipi: SUV
-> Minimum Koltuk: 5
-> Özellikler: geniş bagaj, arazi
-> Bütçe: ekonomik


In [2]:
# @title Entegrasyon Testi (OpenAI api isteği ile test)
import unittest
import time

class TestAraciBulAIGercekAPI(unittest.TestCase):

    def test_1_basarili_istek_tam_bilgi(self):
        """Senaryo 1: Kullanıcının tüm detayları verdiği ideal durum."""
        print("\n--- Gerçek API Test 1: Tam Bilgi ---")
        metin = "Bu hafta sonu 5 kişilik ailemle kampa gideceğim, dağ yollarına uygun, geniş bagajlı ve bütçe dostu bir araç arıyorum."
        sonuc = filtreleme_uygun_arac(metin)

        # Sonucun None dönmediğini ve sözlük (dict) tipinde bir JSON olduğunu doğruluyoruz
        self.assertIsNotNone(sonuc, "API'den geçerli bir yanıt alınamadı.")
        self.assertIsInstance(sonuc, dict)

        # Beklenen anahtarların JSON içinde var olup olmadığını kontrol ediyoruz
        self.assertIn("arac_tipi", sonuc)
        self.assertIn("min_koltuk_sayisi", sonuc)
        self.assertIn("ozellikler", sonuc)
        self.assertIn("butce_tercihi", sonuc)

    def test_2_eksik_bilgi_girilmesi(self):
        """Senaryo 2: Kullanıcının sadece tek bir detay verdiği durum."""
        print("\n--- Gerçek API Test 2: Eksik Bilgi ---")
        metin = "Sadece ucuz bir araba kiralamak istiyorum, gerisi önemli değil."
        sonuc = filtreleme_uygun_arac(metin)

        self.assertIsNotNone(sonuc)
        # Yapay zeka JSON formatını bozmamalı, ancak eksik bilgileri boş bırakmalı veya kendi inisiyatifiyle doldurmalı
        self.assertIsInstance(sonuc, dict)
        # Bütçe tercihini doğru anlayıp anlamadığını kontrol ediyoruz
        bütçe = sonuc.get("butce_tercihi", "").lower()
        self.assertIn("ekonomik", bütçe)

    def test_3_ilgisiz_metin_girilmesi(self):
        """Senaryo 3: Kullanıcının sistemin amacı dışında, tamamen alakasız bir metin girdiği durum."""
        print("\n--- Gerçek API Test 3: İlgisiz Metin ---")
        metin = "Bana güzel bir fırında makarna tarifi verebilir misin?"
        sonuc = filtreleme_uygun_arac(metin)

        self.assertIsNotNone(sonuc)
        self.assertIsInstance(sonuc, dict)
        # Model prompt komutlarına uyup yemek tarifi vermek yerine araçla ilgili boş veya varsayılan bir JSON dönmelidir
        print("İlgisiz metne verilen AI tepkisi:", sonuc)

    def test_4_api_yanit_suresi_performans(self):
        """Senaryo 4: API'nin kullanıcıyı çok fazla bekletmeden yanıt verdiğini test etme (Performans)."""
        print("\n--- Gerçek API Test 4: Performans/Süre Testi ---")
        baslangic_zamani = time.time()

        metin = "Şehir içi kullanım için küçük, otomatik vites bir araç."
        sonuc = filtreleme_uygun_arac(metin)

        bitis_zamani = time.time()
        gecen_sure = bitis_zamani - baslangic_zamani
        print(f"API Yanıt Süresi: {gecen_sure:.2f} saniye")

        self.assertIsNotNone(sonuc)
        # Yanıtın 10 saniyenin altında gelmesi beklenir (İdeal kiralama deneyimi için)
        self.assertLess(gecen_sure, 10.0, "API çok yavaş yanıt verdi, sınır aşıldı.")

# Testleri çalıştır
if __name__ == '__main__':
    unittest.main(argv=['first-arg-is-ignored'], exit=False)


--- Gerçek API Test 1: Tam Bilgi ---
OpenAI API'sine istek gönderiliyor..


.

✅ Başarılı! İşlenmiş Veri / Veritabanı Araması İçin Hazır:
-> Araç Tipi: SUV
-> Minimum Koltuk: 5
-> Özellikler: geniş bagaj, arazi
-> Bütçe: ekonomik

--- Gerçek API Test 2: Eksik Bilgi ---
OpenAI API'sine istek gönderiliyor..


.

✅ Başarılı! İşlenmiş Veri / Veritabanı Araması İçin Hazır:
-> Araç Tipi: herhangi
-> Minimum Koltuk: 1
-> Özellikler: 
-> Bütçe: ekonomik

--- Gerçek API Test 3: İlgisiz Metin ---
OpenAI API'sine istek gönderiliyor..


.

✅ Başarılı! İşlenmiş Veri / Veritabanı Araması İçin Hazır:
-> Araç Tipi: 
-> Minimum Koltuk: 0
-> Özellikler: 
-> Bütçe: 
İlgisiz metne verilen AI tepkisi: {'arac_tipi': '', 'min_koltuk_sayisi': 0, 'ozellikler': [], 'butce_tercihi': ''}

--- Gerçek API Test 4: Performans/Süre Testi ---
OpenAI API'sine istek gönderiliyor..


.
----------------------------------------------------------------------
Ran 4 tests in 4.351s

OK


✅ Başarılı! İşlenmiş Veri / Veritabanı Araması İçin Hazır:
-> Araç Tipi: küçük
-> Minimum Koltuk: 2
-> Özellikler: otomatik vites
-> Bütçe: standart
API Yanıt Süresi: 1.16 saniye


In [3]:
# @title Mocking Testi (Taklit, örnek senaryo çıktıları ile test)
import unittest
from unittest.mock import patch, MagicMock
import json

class TestAraciBulAIMockingGelistirilmis(unittest.TestCase):

    # --------------------------------------------------------------------------
    # SENARYO 1: Tam ve Doğru Bilgi Girilmesi (İdeal Durum)
    # --------------------------------------------------------------------------
    @patch('__main__.client.chat.completions.create')
    def test_1_tam_bilgi_basarili_parse(self, mock_create):
        print("\n --- Mock Test 1: Tam Bilgi Senaryosu ---")
        # API'nin vereceğini varsaydığımız çıktı
        beklenen_json = '{"arac_tipi": "SUV", "min_koltuk_sayisi": 5, "ozellikler": ["geniş bagaj", "arazi"], "butce_tercihi": "ekonomik"}'

        mock_response = MagicMock()
        mock_response.choices[0].message.content = beklenen_json
        mock_create.return_value = mock_response

        sonuc = filtreleme_uygun_arac("5 kişilik, araziye uygun ucuz bir araç lazım.")

        self.assertIsNotNone(sonuc)
        self.assertEqual(sonuc["arac_tipi"], "SUV")
        self.assertEqual(sonuc["min_koltuk_sayisi"], 5)
        print("Mock Test 1 Başarılı: İdeal veri doğru şekilde parse edildi.")

    # --------------------------------------------------------------------------
    # SENARYO 2: Eksik Bilgi Girilmesi
    # --------------------------------------------------------------------------
    @patch('__main__.client.chat.completions.create')
    def test_2_eksik_bilgi(self, mock_create):
        print("\n --- Mock Test 2: Eksik Bilgi Senaryosu ---")
        # Kullanıcı sadece "ucuz araç" dediğinde yapay zekanın döneceği varsayılan yanıt
        beklenen_json = '{"arac_tipi": null, "min_koltuk_sayisi": null, "ozellikler": [], "butce_tercihi": "ekonomik"}'

        mock_response = MagicMock()
        mock_response.choices[0].message.content = beklenen_json
        mock_create.return_value = mock_response

        sonuc = filtreleme_uygun_arac("Sadece ucuz bir araç arıyorum.")

        self.assertIsNotNone(sonuc)
        self.assertEqual(sonuc["butce_tercihi"], "ekonomik")
        self.assertIsNone(sonuc["arac_tipi"]) # Diğer özellikler boş kalmalı
        print("Mock Test 2 Başarılı: Eksik bilgi json formatını bozmadan işlendi.")

    # --------------------------------------------------------------------------
    # SENARYO 3: İlgisiz Metin Girilmesi (Yapay Zekanın Boş Dönmesi)
    # --------------------------------------------------------------------------
    @patch('__main__.client.chat.completions.create')
    def test_3_ilgisiz_metin(self, mock_create):
        print("\n --- Mock Test 3: İlgisiz Metin Senaryosu ---")
        # Makarna tarifi istendiğinde yapay zekanın komutlara uyup boş alanlar dönmesi taklidi
        beklenen_json = '{"arac_tipi": "Belirsiz", "min_koltuk_sayisi": 0, "ozellikler": [], "butce_tercihi": "Belirsiz"}'

        mock_response = MagicMock()
        mock_response.choices[0].message.content = beklenen_json
        mock_create.return_value = mock_response

        sonuc = filtreleme_uygun_arac("Bana fırında makarna tarifi ver.")

        self.assertIsNotNone(sonuc)
        self.assertEqual(sonuc["arac_tipi"], "Belirsiz")
        print("Mock Test 3 Başarılı: İlgisiz metinde güvenlik ve format korundu.")

    # --------------------------------------------------------------------------
    # SENARYO 4: Yapay Zekanın JSON Formatını Bozması (Try-Catch Testi)
    # --------------------------------------------------------------------------
    @patch('__main__.client.chat.completions.create')
    def test_4_hatali_json_formatinda_cokmeme(self, mock_create):
        print("\n --- Mock Test 4: Hatalı JSON/Format Bozulması ---")
        # API'nin JSON kurallarına uymayan düz bir metin döndüğünü varsayıyoruz
        mock_response = MagicMock()
        mock_response.choices[0].message.content = "Üzgünüm, isteğinizi anlayamadım."
        mock_create.return_value = mock_response

        sonuc = filtreleme_uygun_arac("Garip bir istek.")

        # json.loads patlayacak ve except bloğuna düşüp None dönecek. Sistem çökmeyecek.
        self.assertIsNone(sonuc)
        print("Mock Test 4 Başarılı: Bozuk API yanıtında try-catch sistemi korudu.")

    # --------------------------------------------------------------------------
    # SENARYO 5: Ağ veya Sunucu Hatası (Timeout / Exception)
    # --------------------------------------------------------------------------
    @patch('__main__.client.chat.completions.create')
    def test_5_api_baglanti_hatasi(self, mock_create):
        print("\n --- Mock Test 5: API Sunucu Hatası (Exception) ---")
        # İnternet koptuğunda veya OpenAI sunucuları çöktüğünde fırlatılacak hatayı taklit ediyoruz
        mock_create.side_effect = Exception("OpenAI Sunucusuna Ulaşılamıyor veya Timeout!")

        sonuc = filtreleme_uygun_arac("Herhangi bir istek")

        # Sistem yine çökmek yerine bizim belirlediğimiz None değerini dönmeli
        self.assertIsNone(sonuc)
        print("Mock Test 5 Başarılı: Sunucu çökme durumunda try-catch sistemi korudu.")

if __name__ == '__main__':
    unittest.main(argv=['first-arg-is-ignored'], exit=False)

.....
----------------------------------------------------------------------
Ran 5 tests in 0.020s

OK



 --- Mock Test 1: Tam Bilgi Senaryosu ---
OpenAI API'sine istek gönderiliyor..
✅ Başarılı! İşlenmiş Veri / Veritabanı Araması İçin Hazır:
-> Araç Tipi: SUV
-> Minimum Koltuk: 5
-> Özellikler: geniş bagaj, arazi
-> Bütçe: ekonomik
Mock Test 1 Başarılı: İdeal veri doğru şekilde parse edildi.

 --- Mock Test 2: Eksik Bilgi Senaryosu ---
OpenAI API'sine istek gönderiliyor..
✅ Başarılı! İşlenmiş Veri / Veritabanı Araması İçin Hazır:
-> Araç Tipi: None
-> Minimum Koltuk: None
-> Özellikler: 
-> Bütçe: ekonomik
Mock Test 2 Başarılı: Eksik bilgi json formatını bozmadan işlendi.

 --- Mock Test 3: İlgisiz Metin Senaryosu ---
OpenAI API'sine istek gönderiliyor..
✅ Başarılı! İşlenmiş Veri / Veritabanı Araması İçin Hazır:
-> Araç Tipi: Belirsiz
-> Minimum Koltuk: 0
-> Özellikler: 
-> Bütçe: Belirsiz
Mock Test 3 Başarılı: İlgisiz metinde güvenlik ve format korundu.

 --- Mock Test 4: Hatalı JSON/Format Bozulması ---
OpenAI API'sine istek gönderiliyor..
❌ Hata: API'den dönen yanıt geçerli bir JSON 